# 📊 Customer Churn Prediction — Complete Data Pipeline
### Auto Kaggle Download → Cleaning → EDA (10 Charts) → Feature Engineering → Model

| | |
|---|---|
| **Dataset** | Telco Customer Churn (Kaggle) |
| **Domain** | Telecom / Binary Classification |
| **Tools** | Python · Pandas · NumPy · Scikit-learn · Matplotlib · Seaborn · imbalanced-learn |

> ✅ **No manual file upload needed.** Just paste your Kaggle username and API key in Section 1 and run all cells.
---


## 🔑 Section 1 — Kaggle Credentials (Edit Only This Cell)

In [ ]:
# ════════════════════════════════════════════════════════
#  ONLY EDIT THIS CELL
#  Get your credentials from:
#  https://www.kaggle.com/settings  →  API  →  "Create New Token"
#  That downloads kaggle.json — open it and copy the two values below.
# ════════════════════════════════════════════════════════

KAGGLE_USERNAME = "your_kaggle_username"   # ← replace
KAGGLE_KEY      = "your_kaggle_api_key"    # ← replace

# ── Auto-inject credentials into environment ──────────────────────────────────
import os, json, pathlib

os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"]      = KAGGLE_KEY

# Also write kaggle.json so the CLI works
kaggle_dir = pathlib.Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
kaggle_json = kaggle_dir / "kaggle.json"
kaggle_json.write_text(json.dumps({
    "username": KAGGLE_USERNAME,
    "key":      KAGGLE_KEY
}))
kaggle_json.chmod(0o600)

print("✓ Kaggle credentials injected automatically")
print(f"  Username : {KAGGLE_USERNAME}")
print(f"  Key      : {KAGGLE_KEY[:6]}{'*' * (len(KAGGLE_KEY)-6)}")


## 📦 Section 2 — Install Libraries & Download Dataset

In [ ]:
# Install all required libraries
!pip install -q kaggle imbalanced-learn scikit-learn pandas numpy matplotlib seaborn
print("✓ Libraries installed")


In [ ]:
# ── Auto-download dataset from Kaggle ─────────────────────────────────────────
import os

DATASET   = "blastchar/telco-customer-churn"
DATA_DIR  = "./data"
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Downloading dataset: {DATASET}")
os.system(f"kaggle datasets download -d {DATASET} --unzip -p {DATA_DIR}")

# Verify download
files_downloaded = os.listdir(DATA_DIR)
print(f"✓ Files downloaded → {DATA_DIR}/")
for f in files_downloaded:
    size_kb = os.path.getsize(f"{DATA_DIR}/{f}") / 1024
    print(f"   {f}  ({size_kb:.1f} KB)")


## 🔧 Section 3 — Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, pickle, json

from sklearn.pipeline        import Pipeline
from sklearn.compose         import ColumnTransformer
from sklearn.preprocessing   import StandardScaler, OrdinalEncoder, OneHotEncoder, LabelEncoder
from sklearn.impute          import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model    import LogisticRegression
from sklearn.metrics         import (classification_report, confusion_matrix,
                                     roc_auc_score, roc_curve, ConfusionMatrixDisplay)
from imblearn.over_sampling  import SMOTE

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 25)
pd.set_option("display.float_format", "{:.4f}".format)

# ── Plot theme ────────────────────────────────────────────────────────────────
CLR_RETAIN = "#4A90D9"
CLR_CHURN  = "#E24B4A"
CLR_MAIN   = "#534AB7"
PALETTE    = {"No": CLR_RETAIN, "Yes": CLR_CHURN}

plt.rcParams.update({
    "figure.dpi"       : 130,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
})

CHARTS_DIR  = "./charts"
OUTPUTS_DIR = "./outputs"
os.makedirs(CHARTS_DIR,  exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print("✓ All imports done")
print(f"✓ Charts  → {CHARTS_DIR}/")
print(f"✓ Outputs → {OUTPUTS_DIR}/")


---
## 📂 Section 4 — Raw Data Ingestion & First Look

In [ ]:
# ── Auto-detect CSV file in data dir ─────────────────────────────────────────
import glob

csv_files = glob.glob("./data/*.csv")
if not csv_files:
    raise FileNotFoundError("No CSV found in ./data/ — check Section 1 credentials.")

CSV_PATH = csv_files[0]
print(f"Loading: {CSV_PATH}")

df_raw = pd.read_csv(CSV_PATH)

print("=" * 55)
print(f"  SHAPE : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print("=" * 55)
df_raw.head(8)


In [ ]:
# ── Column inventory ─────────────────────────────────────────────────────────
print(f"{'Column':<25} {'Dtype':<15} {'Unique':>8} {'Nulls':>8}")
print("-" * 60)
for col in df_raw.columns:
    print(f"  {col:<23} {str(df_raw[col].dtype):<15} "
          f"{df_raw[col].nunique():>8} {df_raw[col].isnull().sum():>8}")


In [ ]:
df_raw.describe(include="all").T


---
## 🧹 Section 5 — Data Cleaning

In [ ]:
df = df_raw.copy()

# ── 5.1 Fix TotalCharges: whitespace strings → float ─────────────────────────
print("Before fix → TotalCharges dtype:", df["TotalCharges"].dtype)
bad_rows = (df["TotalCharges"].str.strip() == "").sum()
print(f"Whitespace/empty entries: {bad_rows}")

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].str.strip(), errors="coerce")

print("After fix  → TotalCharges dtype:", df["TotalCharges"].dtype)
print(f"NaN count after coerce          : {df['TotalCharges'].isnull().sum()}")


In [ ]:
# ── 5.2 Impute TotalCharges NaNs (new customers with 0 tenure) ──────────────
median_tc = df["TotalCharges"].median()
df["TotalCharges"].fillna(median_tc, inplace=True)
print(f"✓ Imputed {bad_rows} TotalCharges nulls with median = {median_tc:.2f}")

# ── 5.3 Drop non-informative ID column ───────────────────────────────────────
df.drop(columns=["customerID"], inplace=True)
print("✓ Dropped customerID")

# ── 5.4 Remove duplicates ────────────────────────────────────────────────────
n_dup = df.duplicated().sum()
if n_dup > 0:
    df.drop_duplicates(inplace=True)
    print(f"✓ Removed {n_dup} duplicate rows")
else:
    print("✓ No duplicates found")

# ── 5.5 Standardise 'No X service' → 'No' across service columns ────────────
service_cols_fix = [
    "MultipleLines","OnlineSecurity","OnlineBackup",
    "DeviceProtection","TechSupport","StreamingTV","StreamingMovies"
]
df[service_cols_fix] = df[service_cols_fix].replace(
    {"No phone service": "No", "No internet service": "No"}
)
print("✓ Standardised 'No phone/internet service' → 'No'")

# ── 5.6 Encode target ────────────────────────────────────────────────────────
df["Churn_Flag"] = (df["Churn"] == "Yes").astype(int)

print("\n── CLEANING COMPLETE ──────────────────────────────────")
print(f"  Shape          : {df.shape}")
print(f"  Nulls          : {df.isnull().sum().sum()}")
print(f"  Churn rate     : {df['Churn_Flag'].mean()*100:.2f}%")
print(f"  Churned        : {df['Churn_Flag'].sum():,}")
print(f"  Retained       : {(df['Churn_Flag']==0).sum():,}")


---
## 📊 Section 6 — Exploratory Data Analysis (10 Charts)

In [ ]:
# ── Chart 1 : Overall Churn Rate ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
counts = df["Churn"].value_counts()

axes[0].bar(counts.index, counts.values,
            color=[CLR_RETAIN, CLR_CHURN], width=0.5, edgecolor="white")
axes[0].set_title("Customer count by churn", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Count")
for i,(idx,v) in enumerate(counts.items()):
    axes[0].text(i, v+60, f"{v:,}\n({v/len(df)*100:.1f}%)", ha="center", fontsize=10)

axes[1].pie(counts.values, labels=counts.index, colors=[CLR_RETAIN, CLR_CHURN],
            autopct="%1.1f%%", startangle=90,
            wedgeprops=dict(width=0.55, edgecolor="white", linewidth=2))
axes[1].set_title("Churn proportion (donut)", fontsize=12, fontweight="bold")

plt.suptitle("Chart 1 — Overall Churn Rate", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{CHARTS_DIR}/chart_01_churn_rate.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Churn rate: {df['Churn_Flag'].mean()*100:.2f}%  |  Imbalance: {(df['Churn']=='No').sum()/(df['Churn']=='Yes').sum():.1f}:1")


In [ ]:
# ── Chart 2 : Tenure Distribution by Churn ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (label, grp) in zip(axes, df.groupby("Churn")["tenure"]):
    color = CLR_CHURN if label == "Yes" else CLR_RETAIN
    ax.hist(grp, bins=30, color=color, alpha=0.85, edgecolor="white")
    ax.set_title(f"Tenure — Churn = {label}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Tenure (months)")
    ax.set_ylabel("Count")
    ax.axvline(grp.mean(), color="black", linestyle="--", lw=1.5,
               label=f"Mean: {grp.mean():.1f}m")
    ax.legend()
plt.suptitle("Chart 2 — Tenure Distribution by Churn", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{CHARTS_DIR}/chart_02_tenure.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Chart 3 : Monthly Charges Violin ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
parts = ax.violinplot(
    [df[df["Churn"]==c]["MonthlyCharges"].values for c in ["No","Yes"]],
    positions=[1,2], showmedians=True
)
for pc, color in zip(parts["bodies"], [CLR_RETAIN, CLR_CHURN]):
    pc.set_facecolor(color); pc.set_alpha(0.7)
parts["cmedians"].set_color("black"); parts["cmedians"].set_linewidth(2)
ax.set_xticks([1,2]); ax.set_xticklabels(["Retained","Churned"], fontsize=12)
ax.set_ylabel("Monthly Charges ($)")
ax.set_title("Chart 3 — Monthly Charges by Churn", fontsize=14, fontweight="bold")
for i,c in enumerate(["No","Yes"],1):
    med = df[df["Churn"]==c]["MonthlyCharges"].median()
    ax.text(i, med+2, f"${med:.0f}", ha="center", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CHARTS_DIR}/chart_03_monthly_charges.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Chart 4 : Churn Rate by Contract Type ────────────────────────────────────
ct = (df.groupby("Contract")["Churn_Flag"]
        .agg(["mean","count"])
        .rename(columns={"mean":"Rate","count":"N"})
        .reset_index())
ct["Rate_pct"] = ct["Rate"] * 100

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(ct["Contract"], ct["Rate_pct"],
              color=[CLR_CHURN, CLR_RETAIN, "#7DC67D"], width=0.5, edgecolor="white")
ax.set_ylabel("Churn Rate (%)"); ax.set_ylim(0, 55)
ax.set_title("Chart 4 — Churn Rate by Contract Type", fontsize=14, fontweight="bold")
for bar, (_, row) in zip(bars, ct.iterrows()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.8,
            f"{row.Rate_pct:.1f}%\n(n={row.N:,})", ha="center", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CHARTS_DIR}/chart_04_contract.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Chart 5 : Service Subscription Heatmap ───────────────────────────────────
svc = ["PhoneService","MultipleLines","OnlineSecurity","OnlineBackup",
       "DeviceProtection","TechSupport","StreamingTV","StreamingMovies"]
heat = pd.DataFrame({c: df.groupby(c)["Churn_Flag"].mean()*100 for c in svc}).T
heat.columns = [f"Retained %" if c=="No" else "Churned %" for c in heat.columns]

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(heat, annot=True, fmt=".1f", cmap="RdYlGn_r",
            linewidths=0.5, linecolor="white",
            cbar_kws={"label":"Churn rate (%)"}, ax=ax)
ax.set_title("Chart 5 — Churn Rate by Service Subscription", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CHARTS_DIR}/chart_05_service_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Chart 6 : Feature Correlation with Churn ─────────────────────────────────
df_enc = df.copy()
for col in df_enc.select_dtypes("object").columns:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col])

corr = df_enc.corr()["Churn_Flag"].drop(["Churn_Flag","Churn"]).sort_values()
fig, ax = plt.subplots(figsize=(8, 7))
colors = [CLR_CHURN if v > 0 else CLR_RETAIN for v in corr.values]
ax.barh(corr.index, corr.values, color=colors, edgecolor="white")
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Pearson correlation with Churn")
ax.set_title("Chart 6 — Feature Correlation with Churn", fontsize=14, fontweight="bold")
for i, v in enumerate(corr.values):
    ax.text(v+(0.003 if v>=0 else -0.003), i, f"{v:.3f}", va="center",
            ha="left" if v>=0 else "right", fontsize=8.5)
plt.tight_layout()
plt.savefig(f"{CHARTS_DIR}/chart_06_correlation.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Chart 7 : Internet Service & Payment Method ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

internet = df.groupby(["InternetService","Churn"]).size().unstack(fill_value=0)
internet.div(internet.sum(axis=1), axis=0).mul(100).plot(
    kind="bar", ax=axes[0], color=[CLR_RETAIN, CLR_CHURN], edgecolor="white", rot=15)
axes[0].set_title("Internet service vs churn %", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Percentage (%)"); axes[0].legend(title="Churn")

pay = (df.groupby("PaymentMethod")["Churn_Flag"].mean()*100).sort_values()
axes[1].barh(pay.index, pay.values, color=CLR_MAIN, edgecolor="white")
axes[1].set_xlabel("Churn Rate (%)")
axes[1].set_title("Payment method churn rate", fontsize=12, fontweight="bold")
for i, v in enumerate(pay.values):
    axes[1].text(v+0.3, i, f"{v:.1f}%", va="center", fontsize=10)

plt.suptitle("Chart 7 — Internet Service & Payment Method", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{CHARTS_DIR}/chart_07_internet_payment.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Chart 8 : Tenure Bins Dual-Axis ──────────────────────────────────────────
df["Tenure_Bin"] = pd.cut(df["tenure"], bins=[0,12,24,48,72],
                           labels=["0-12m","13-24m","25-48m","49-72m"])
tb = df.groupby("Tenure_Bin", observed=True)["Churn_Flag"].agg(["mean","count"]).reset_index()
tb["pct"] = tb["mean"] * 100

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()
ax1.bar(tb["Tenure_Bin"].astype(str), tb["count"],
        color=CLR_RETAIN, alpha=0.6, width=0.5, label="Customer count")
ax2.plot(tb["Tenure_Bin"].astype(str), tb["pct"],
         color=CLR_CHURN, marker="o", lw=2.5, ms=8, label="Churn rate %")
ax1.set_ylabel("Number of customers"); ax2.set_ylabel("Churn rate (%)", color=CLR_CHURN)
ax2.tick_params(axis="y", labelcolor=CLR_CHURN)
ax1.set_title("Chart 8 — Customer Count & Churn by Tenure Bin", fontsize=13, fontweight="bold")
lines = ax1.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
labs  = ax1.get_legend_handles_labels()[1] + ax2.get_legend_handles_labels()[1]
ax1.legend(lines, labs, loc="upper right")
plt.tight_layout()
plt.savefig(f"{CHARTS_DIR}/chart_08_tenure_bins.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Chart 9 : Tenure vs Monthly Charges Scatter ──────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
for label, color in [("No", CLR_RETAIN), ("Yes", CLR_CHURN)]:
    sub = df[df["Churn"]==label]
    ax.scatter(sub["tenure"], sub["MonthlyCharges"],
               c=color, alpha=0.25, s=15, label=f"Churn={label} (n={len(sub):,})")
ax.set_xlabel("Tenure (months)"); ax.set_ylabel("Monthly Charges ($)")
ax.set_title("Chart 9 — Tenure vs Monthly Charges (by Churn)", fontsize=13, fontweight="bold")
ax.legend(markerscale=2)
plt.tight_layout()
plt.savefig(f"{CHARTS_DIR}/chart_09_scatter.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Chart 10 : Pairplot of Numeric Features ──────────────────────────────────
g = sns.pairplot(df[["tenure","MonthlyCharges","TotalCharges","Churn"]],
                 hue="Churn", palette=PALETTE,
                 plot_kws={"alpha":0.3,"s":15},
                 diag_kind="kde", corner=True)
g.fig.suptitle("Chart 10 — Pairplot: Numeric Features by Churn",
               y=1.02, fontsize=14, fontweight="bold")
g.fig.savefig(f"{CHARTS_DIR}/chart_10_pairplot.png", dpi=130, bbox_inches="tight")
plt.show()
print("✓ All 10 EDA charts saved to ./charts/")


---
## ⚙️ Section 7 — Feature Engineering

In [ ]:
df_feat = df.copy()

# 1. Revenue per month
df_feat["Revenue_PerMonth"] = np.where(
    df_feat["tenure"] > 0,
    df_feat["TotalCharges"] / df_feat["tenure"],
    df_feat["MonthlyCharges"]
)

# 2. Number of add-on services subscribed
add_ons = ["OnlineSecurity","OnlineBackup","DeviceProtection",
           "TechSupport","StreamingTV","StreamingMovies"]
df_feat["Services_Count"] = (df_feat[add_ons] == "Yes").sum(axis=1)

# 3. Is new customer (≤ 12 months)
df_feat["Is_NewCustomer"] = (df_feat["tenure"] <= 12).astype(int)

# 4. Is long-term customer (≥ 48 months)
df_feat["Is_LongTerm"] = (df_feat["tenure"] >= 48).astype(int)

# 5. Charge per service unit
df_feat["Charge_PerService"] = df_feat["MonthlyCharges"] / (df_feat["Services_Count"] + 1)

# 6. Has any streaming service
df_feat["Has_Streaming"] = (
    (df_feat["StreamingTV"]=="Yes") | (df_feat["StreamingMovies"]=="Yes")
).astype(int)

# 7. Has full security bundle (both backup + security)
df_feat["Has_SecurityBundle"] = (
    (df_feat["OnlineSecurity"]=="Yes") & (df_feat["OnlineBackup"]=="Yes")
).astype(int)

engineered = ["Revenue_PerMonth","Services_Count","Is_NewCustomer",
              "Is_LongTerm","Charge_PerService","Has_Streaming","Has_SecurityBundle"]

print("ENGINEERED FEATURES")
print("="*55)
for f in engineered:
    print(f"  {f:<25}: range [{df_feat[f].min():.2f}, {df_feat[f].max():.2f}]")

df_feat[engineered + ["Churn_Flag"]].describe()
